# Saved (named) LLM configs

A **saved LLM config** is a team-scoped, named record wrapping either a
single provider config or an LLM group. Instead of inlining provider
settings (and API keys) into every node, you register the config once
and reference it from a node via `attachable_llm_config_id`.

Everything lives under `client.llm_configs`: `schema`, `create`,
`list`, `get_default`, `get`, `update`, `delete`, `test`, `test_inline`,
`export`, `import_bundle`, `usage`.

See the companion guide: [`../docs/guides/llm_configs.md`](../docs/guides/llm_configs.md).

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from interactly import AsyncWorkflowClient
from interactly.configs import OpenAILLMConfig, OPENAIModel

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

In [ ]:
# Idempotency guard — remove anything left over from a previous run so that
# re-running this notebook always starts from a clean slate (config names must be unique).
_demo_names = {"Primary GPT config", "Primary GPT config (v2)"}
_gateway_name = "Notebook gateway (10_llm_configs)"  # and its "(imported)" copies
_existing = await (await client.llm_configs.list(size=200)).list_all()

# This notebook marks its demo config as the team default (override_default=True takes the flag away from
# whatever held it). Remember the real default so the Cleanup cell can give it back; the demo config is
# deleted at the end, and without this the team would be left with no default at all.
_previous_default_id = next((c.id for c in _existing if c.is_default and c.name not in _demo_names), None)
print("team default before this notebook:", _previous_default_id)
for _c in _existing:
    if _c.name in _demo_names or _c.name.startswith(_gateway_name):
        await client.llm_configs.delete(_c.id)

# Also remove the demo workflow this notebook builds at the end (see "Referencing a
# saved config from a node"), in case a prior run was interrupted before cleanup.
_demo_wf_name = "Saved-config demo (10_llm_configs)"
for _w in await (await client.workflows.list(size=200)).list_all():
    if _w.name == _demo_wf_name:
        await client.workflows.delete(_w.id)

print("Cleared any pre-existing demo assets.")

## Discover the schema

`schema()` returns the JSON Schema for a saved config's `config` field
(an `LLMOrGroupConfig`).

In [ ]:
from typing import Dict, Any

schema: Dict[str, Any] = await client.llm_configs.schema()
print(sorted(schema.keys()))

## Create a saved config

`create()` takes a keyword-only `name` and `config`. The `config` can be
a typed `OpenAILLMConfig` (or any `LLMOrGroupConfig`) or a compatible
dict. Set `is_default=True` to mark it as the team default; pair with
`override_default=True` to replace an existing default.

A team has at most one default, so `override_default=True` takes the flag **away from the config that held it**; it does not come back when this config is deleted. This notebook remembers the team's default in its setup cell and restores it in Cleanup. Do the same in your own code.

In [ ]:
cfg = OpenAILLMConfig(
    model=OPENAIModel.GPT_5_4,
    max_tokens=8192,
    temperature=0.2,
)

saved = await client.llm_configs.create(
    name="Primary GPT config",
    config=cfg,
    description="Default OpenAI settings for assistant nodes",
    is_default=True,
    override_default=True,
)
print(saved.id, saved.name, saved.is_default)

## List, get, and fetch the default

In [ ]:
# Paginated list with optional fuzzy `search`.
page = await client.llm_configs.list(size=10, search="GPT")
for item in page.items:
    print(item.id, item.name, item.is_default)

# Fetch a specific config by id.
fetched = await client.llm_configs.get(saved.id)

# Fetch whichever config the team has marked as default.
default_cfg = await client.llm_configs.get_default()
print(default_cfg.id, default_cfg.name)

## Update

`update()` sends only the fields you pass, but `config` replaces the stored config **as a whole**.

Whether you are shown the stored credentials depends on your role: admins and super-admins see them; every
other role reads `api_key` and the AWS keys as `None`, and credential headers as `""`. For an admin, the
server reads a credential the config leaves blank as "remove it". A config built fresh in code, like the
one below, has none. So `update()` keeps the stored key by
default (`preserve_credentials=True`): it reads the stored config first and carries across each credential
`config` leaves blank (`api_key`, Bedrock's AWS keys, a gateway's credential headers). Pass
`preserve_credentials=False` to remove them on purpose.

In [ ]:
updated = await client.llm_configs.update(
    saved.id,
    name="Primary GPT config (v2)",
    description="Bumped temperature",
    config=OpenAILLMConfig(model=OPENAIModel.GPT_5_4, max_tokens=8192, temperature=0.4),
)
print(updated.name)

### Credentials an update keeps

The OpenAI config above holds no key of its own, so there is nothing to keep. The cell below uses a config that does: a custom gateway with a made-up `api_key` and `Authorization` header. It points at a host that can never resolve, so nothing here reaches a provider.

The update sends a config with neither. The SDK reads the stored config first and carries both across. Only credential headers are carried: `X-Tenant` is an ordinary header the new config left out, so it is gone.

`api_key` is a `SecretStr`, masked when printed; `get_secret_value()` reads it. What it reads depends on your role. An admin's key shows the stored values; any other role reads the key as `None` and the header as `""`, though both are kept.

In [ ]:
from interactly.configs import CustomLLMConfig

GATEWAY_NAME = "Notebook gateway (10_llm_configs)"
FAKE_KEY = "sk-notebook-not-a-secret"
FAKE_AUTH = "Bearer notebook-not-a-secret"


def gateway(**credentials) -> CustomLLMConfig:
    """A gateway at a host that can never resolve, so nothing here calls a provider."""
    return CustomLLMConfig(
        base_url="https://llm.unreachable.invalid/v1",
        model="none",
        max_retries=0,
        request_timeout_ms=2000,
        streaming=False,
        **credentials,
    )


gw = await client.llm_configs.create(
    name=GATEWAY_NAME,
    config=gateway(api_key=FAKE_KEY, default_headers={"Authorization": FAKE_AUTH, "X-Tenant": "t1"}),
)
GATEWAY_ID = gw.id

# An update built in code, with no credentials at all.
await client.llm_configs.update(GATEWAY_ID, config=gateway())

stored = (await client.llm_configs.get(GATEWAY_ID)).config
# api_key is a SecretStr, shown as ********** when printed; get_secret_value() reads it.
print("api_key kept:", stored.api_key.get_secret_value() if stored.api_key else None)
print("headers kept:", stored.default_headers)

## Test a config

Two flavours:

- `test(id, ...)` exercises a **saved** config against a `system_prompt`
  (+ optional `messages`); a redacted inline `config` override reuses the
  stored secret.
- `test_inline(...)` exercises an **unsaved** config before persisting —
  the `config` must carry its own `api_key` (or omit it to fall back to
  the team's stored vendor credentials).

Both return an `LLMConfigTestResult` (`success`, `response`, token counts,
`latency_ms`, plus group extras like `winning_member`).

In [ ]:
# Test the saved config.
result = await client.llm_configs.test(
    saved.id,
    system_prompt="You are a terse assistant. Answer in one short sentence.",
    messages=[{"role": "human", "content": "What is an insurance deductible?"}],
)
print(result.success, result.response)
print(result.provider, result.model, result.total_tokens, result.latency_ms)
print("used the team's own key:", result.used_user_api_key)

# Test an inline (unsaved) config before you decide to persist it.
inline_result = await client.llm_configs.test_inline(
    system_prompt="You are a terse assistant.",
    config=OpenAILLMConfig(model=OPENAIModel.GPT_5_4_NANO, max_tokens=8192),
    messages=[{"role": "human", "content": "Say hello."}],
)
print(inline_result.success, inline_result.response)

**A failed test says what kind of failure.** `error_category` is one of `auth`, `permission`, `quota`, `rate_limit`, `model`, `timeout`, `unavailable` or `other`, and `error` holds the provider's own message. `used_user_api_key` says whether the call used the team's own saved key rather than the deployment's. The gateway above can't be reached, so its test fails as `unavailable`.

In [ ]:
failed = await client.llm_configs.test(
    GATEWAY_ID,
    system_prompt="Say hi.",
    messages=[{"role": "human", "content": "hi"}],
)
print(failed.success, failed.error_category, failed.used_user_api_key)
print(str(failed.error)[:160])

## Providers served through Vertex AI, and backends

Grok, Gemma, GLM and DeepSeek are reached through Google Vertex AI on the platform's Google Cloud
credentials, so **none of them takes an API key**. They answer only on Vertex's global endpoint.

Claude and Gemini can each be served two ways. The choice is about credentials and billing, not the model:

- `AnthropicLLMConfig.backend`: `direct` (the default, api.anthropic.com) or `vertex`. On Vertex only
  some Claude models are available; `vertex_selectable_anthropic_models()` lists them.
- `GoogleLLMConfig.backend`: unset by default, which keeps the server's rule (a team Gemini key means the
  public API, no key means Vertex). Set `vertex` or `ai_studio` to pin one.

Gemini 3 models take a `thinking_level` instead of a token budget. Each test below is one small, real
model call.

In [ ]:
import interactly_configs as ic
from interactly.configs import (
    AnthropicBackend,
    AnthropicLLMConfig,
    ANTHROPICModel,
    GeminiThinkingLevel,
    GoogleBackend,
    GoogleLLMConfig,
    GOOGLEModel,
    XAILLMConfig,
    XAIModel,
)

print("Claude models available on Vertex:", sorted(m.value for m in ic.vertex_selectable_anthropic_models()))

candidates = {
    # Grok's non-reasoning variants are the low-latency choice for voice nodes.
    "Grok (via Vertex)": XAILLMConfig(model=XAIModel.GROK_4_1_FAST_NON_REASONING, max_tokens=8192),
    "Claude on Vertex": AnthropicLLMConfig(
        model=ANTHROPICModel.CLAUDE_SONNET_4_6, backend=AnthropicBackend.VERTEX, max_tokens=8192
    ),
    "Gemini 3.8 on Vertex": GoogleLLMConfig(
        model=GOOGLEModel.GEMINI_3_8_FLASH,
        backend=GoogleBackend.VERTEX,
        thinking_level=GeminiThinkingLevel.LOW,
        max_tokens=8192,
    ),
}
for label, config in candidates.items():
    result = await client.llm_configs.test_inline(
        system_prompt="Reply with one short sentence.",
        messages=[{"role": "human", "content": "Say hello."}],
        config=config,
    )
    print(f"{label:22s} success={result.success}  model={result.model}  {result.response or result.error!r}")

## Export and import

`export()` writes saved configs into a portable bundle. Every secret in it is replaced with `"<REDACTED>"`, whatever your role, and listed in `bundle["redactions"]`. Leave out `llm_config_ids` to export every config of the team.

`import_bundle()` is all or nothing: every redacted secret must come back in `secret_overrides`, keyed by its path, or nothing is created. A name the team already uses gets an `(imported)` suffix (`renamed_from` holds the bundle's name), and an id it already uses is replaced by a new one (`ids_reminted`). Imported configs never become the team default.

In [ ]:
from interactly import UnprocessableEntityError

bundle = await client.llm_configs.export(llm_config_ids=[GATEWAY_ID])
for redaction in bundle["redactions"]:
    print(f"redacted: {redaction['path']}  ({redaction['reason']})")

try:
    await client.llm_configs.import_bundle(bundle)
except UnprocessableEntityError as exc:
    print(f"\nWithout the secrets: {exc}")

values = {"api_key": FAKE_KEY, "Authorization": FAKE_AUTH}
overrides = {r["path"]: values[r["path"].rsplit(".", 1)[-1]] for r in bundle["redactions"]}
import_result = await client.llm_configs.import_bundle(bundle, secret_overrides=overrides)
IMPORTED_LLM_CONFIG_IDS = [item.llm_config.id for item in import_result.imported]
for item in import_result.imported:
    print(f"\nimported {item.llm_config.name!r}")
    print(f"  renamed from: {item.renamed_from!r}   new ids: {item.ids_reminted}")

## Referencing a saved config from a node

Point a node at a saved config with **`attachable_llm_config_id`**, instead of inlining provider settings:

```python
node = SayLLMNodeConfig(
    name="Assistant",
    main_response_config=PromptConfig(prompt="You are a helpful assistant."),
    attachable_llm_config_id=saved.id,
)
```

On every fully-hydrated fetch the server copies the saved config into the node's `llms_config`, so editing
the saved config takes effect without re-publishing the workflow. `named_llm_config_id` and
`named_llm_config_name` on the resolved config are filled in **by the server** to record where it came
from; setting them yourself references nothing.

The cell below reads the workflow back first, so you can see the resolution happen before the run.

In [ ]:
# Build a single-LLM-node workflow whose node takes its provider settings from the SAVED config,
# show the server resolving the reference, then run one turn on it.
from langchain_core.messages import HumanMessage

from interactly.configs import (
    LLMNodeRunInput,
    NodesRunInputs,
    PromptConfig,
    SayLLMNodeConfig,
    WorkflowCommand,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
    WorkflowRunInput,
)
from interactly.runtime.events import AssistantResponseEvent
from interactly.types.workflows.workflow import Workflow

# The node inlines NO provider settings — only the id of the saved config.
assistant_node = SayLLMNodeConfig(
    name="Assistant",
    is_start=True,
    wait_for_user_message=True,
    main_response_config=PromptConfig(
        prompt="You are a helpful health-insurance assistant. Answer the user's question in one short sentence.",
    ),
    attachable_llm_config_id=saved.id,
)

ref_workflow: Workflow = await client.workflows.create_from_config(
    WorkflowConfigFullyHydrated(
        workflow_config=WorkflowConfig(
            name="Saved-config demo (10_llm_configs)",
            description="Single LLM node that takes its provider settings from a saved config",
        ),
        node_configs=[assistant_node],
        edge_configs=[],
    )
)
REF_WF_ID = ref_workflow.id
print(f"Created workflow id={REF_WF_ID}")

# Read it back: the server has copied the saved config into the node.
resolved = (await client.workflows.get_fully_hydrated(REF_WF_ID)).node_configs[0].llms_config
print(f"Resolved llms_config: {type(resolved).__name__}  model={resolved.model.value}")
print(f"  named_llm_config_id={resolved.named_llm_config_id}  (the saved config: {saved.id})")
assert resolved.named_llm_config_id == saved.id, "the node did not resolve to the saved config"

# Run one turn on it.
chat = await client.workflows.handle(REF_WF_ID)
run_input = WorkflowRunInput(
    command=WorkflowCommand.START,
    thread_to_node_inputs={
        "0": NodesRunInputs(
            node_run_inputs=[LLMNodeRunInput(messages=[HumanMessage(content="What is a co-payment?")])]
        )
    },
)
print("User: What is a co-payment?")
async for event in chat.arun(run_input):
    if isinstance(event, AssistantResponseEvent) and event.content:
        print(f"  Assistant: {event.content}")

## Where a saved config is used

`usage()` lists the team's workflows that reference a saved config, in any version, active or not. A delete does not consult it: deleting the config succeeds and leaves those workflows pointing at nothing. The demo workflow above references the saved config.

In [ ]:
report = await client.llm_configs.usage(saved.id)
for entry in report.used_by:
    print(f"used by {entry.name!r} ({entry.workflow_id}), versions {entry.version_numbers}")

## Cleanup

In [ ]:
# Delete the demo workflow, then the saved config it referenced.
await client.workflows.delete(REF_WF_ID)
await client.llm_configs.delete(saved.id)
for llm_config_id in [GATEWAY_ID, *IMPORTED_LLM_CONFIG_IDS]:
    await client.llm_configs.delete(llm_config_id)

# Give the team its default back: creating the demo config with override_default=True took the flag away.
if _previous_default_id:
    await client.llm_configs.update(_previous_default_id, is_default=True, override_default=True)
    print("restored the team default:", _previous_default_id)
print("cleaned up")

## See also

- Guide: [`../docs/guides/llm_configs.md`](../docs/guides/llm_configs.md)
- [`11_reusable_assets.ipynb`](11_reusable_assets.ipynb) — node libraries, templates, categories
- [`02_interactive_workflow.ipynb`](02_interactive_workflow.ipynb) — where `llms_config` lives on a node